# 🎬 Video & Image Enhancer AI — Full Training Notebook

## Before running
1. **Runtime → Change runtime type → A100 GPU** (requires Colab Pro)
2. Run cells top to bottom. Each section is clearly labelled.

## What this notebook does
| Stage | Model | Dataset | Time (A100) |
|---|---|---|---|
| 1 | Image SR (RRDBNet) | DIV2K + Flickr2K | ~2 hours |
| 2 | Video SR | REDS | ~4 hours |
| 3 | Deblur | GoPro | ~1 hour |
| 4 | Joint fine-tune | DIV2K sample | ~30 min |

**Total: ~8 hours.** You can run stages independently and download checkpoints after each one.

## Disk usage
| Dataset | Size |
|---|---|
| DIV2K | ~7 GB |
| Flickr2K | ~20 GB |
| REDS | ~120 GB |
| GoPro | ~6 GB |
| **Total** | **~153 GB** |

Colab Pro gives you ~200 GB. This fits — but only just. We delete each dataset after training its stage to free space.

---
## Step 1: Check GPU and disk

In [ ]:
import torch, os

print('=== GPU ===')
if torch.cuda.is_available():
    print(f'  GPU   : {torch.cuda.get_device_name(0)}')
    print(f'  VRAM  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('  WARNING: No GPU. Go to Runtime > Change runtime type > A100 GPU')

print('\n=== Disk ===')
os.system('df -h /content')

print('\n=== RAM ===')
os.system('free -h')

---
## Step 2: Install dependencies

In [ ]:
!pip install torch torchvision numpy opencv-python scikit-image tqdm lpips -q
print('Dependencies installed.')

---
## Step 3: Upload your project

Run the cell below. When the file picker appears, upload **`video_image_enhancer.zip`**.

To create the zip on Windows:
```powershell
Compress-Archive -Path "C:\Users\...\Desktop\AI handling\video_image_enhancer\*" `
  -DestinationPath "C:\Users\...\Desktop\video_image_enhancer.zip"
```

In [ ]:
from google.colab import files
import zipfile, sys, glob, os

print('Upload video_image_enhancer.zip ...')
uploaded = files.upload()

for fname in uploaded:
    if fname.endswith('.zip'):
        with zipfile.ZipFile(fname, 'r') as z:
            # Fix Windows backslash paths before extracting
            for member in z.infolist():
                member.filename = member.filename.replace('\\', '/')
                z.extract(member, '/content/')
        print(f'Extracted: {fname}')

# ── Find project root robustly ──────────────────────────────────────────
hits = glob.glob('/content/**/enhancer/__init__.py', recursive=True)

if hits:
    PROJECT_ROOT = os.path.dirname(os.path.dirname(hits[0]))
    print(f'Found project at: {PROJECT_ROOT}')
else:
    print('Could not find enhancer/__init__.py. Contents of /content:')
    for item in sorted(os.listdir('/content')):
        print(f'  {item}')
    raise FileNotFoundError('Project not found after extraction.')

os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)

for d in ['checkpoints', 'data', 'training_logs', 'feedback_data']:
    os.makedirs(d, exist_ok=True)

print(f'Working directory: {os.getcwd()}')
print('Ready.')

---
## Step 4: Load Real-ESRGAN pretrained weights (starting point)

In [ ]:
!wget -q --show-progress -O checkpoints/image_sr_realesrgan.pth \
    https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth

!python scripts/load_pretrained.py adapt \
    --src checkpoints/image_sr_realesrgan.pth \
    --dst checkpoints/image_sr.pth

print('Pretrained weights loaded into checkpoints/image_sr.pth')

---
# STAGE 1: Image Super-Resolution
**Dataset:** DIV2K (800 images) + Flickr2K (2650 images)  
**Time:** ~2 hours on A100  
**Improves:** Sharpness, upscaling quality, noise handling on still images

In [ ]:
# Download DIV2K (~7 GB)
print('Downloading DIV2K_train_HR (~7 GB)...')
!wget -q --show-progress -O data/DIV2K_train_HR.zip \
    http://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_train_HR.zip
!unzip -q data/DIV2K_train_HR.zip -d data/
!rm data/DIV2K_train_HR.zip
print(f'DIV2K: {len(glob.glob("data/DIV2K_train_HR/*.png"))} images')

In [ ]:
# Download Flickr2K (~20 GB)
print('Downloading Flickr2K (~20 GB)...')
!wget -q --show-progress -O data/Flickr2K.zip \
    https://cv.snu.ac.kr/research/EDSR/Flickr2K.tar
!tar -xf data/Flickr2K.tar -C data/ 2>/dev/null || unzip -q data/Flickr2K.zip -d data/
!rm -f data/Flickr2K.zip data/Flickr2K.tar

# Combine into one folder
!mkdir -p data/ImageSR_combined
!cp data/DIV2K_train_HR/*.png data/ImageSR_combined/ 2>/dev/null || true
!find data/ -name '*.png' -path '*/Flickr2K/*' -exec cp {} data/ImageSR_combined/ \; 2>/dev/null || true
print(f'Combined image SR dataset: {len(glob.glob("data/ImageSR_combined/*.png"))} images')

In [ ]:
import torch, torch.nn as nn, torch.optim as optim
import numpy as np, time, json
import torch.nn.functional as F
from torch.utils.data import DataLoader

from enhancer.models.rrdb_net import RRDBNet
from scripts.train import SyntheticDegradationDataset, CombinedSRLoss, AuditLogger

EPOCHS_SR  = 100
BATCH_SIZE = 16    # A100 can handle larger batches
LR_SR      = 1e-5  # fine-tuning from pretrained
PATCH_SIZE = 128

device = torch.device('cuda')
model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=4).to(device)
state = torch.load('checkpoints/image_sr.pth', map_location='cpu', weights_only=False)
model.load_state_dict(state, strict=False)
model.train()

dataset   = SyntheticDegradationDataset('data/ImageSR_combined', scale=4, patch_size=PATCH_SIZE)
loader    = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=8, pin_memory=True)
optimizer = optim.Adam(model.parameters(), lr=LR_SR)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_SR)
loss_fn   = CombinedSRLoss(l1_weight=1.0, perceptual_weight=0.05)
audit     = AuditLogger('training_logs')

best_loss = float('inf')
t0 = time.time()
print(f'Stage 1: Image SR | {len(dataset)} samples | {EPOCHS_SR} epochs | batch={BATCH_SIZE}')
print(f'Estimated time: ~{EPOCHS_SR * len(loader) * 0.03 / 60:.0f} minutes')

In [ ]:
for epoch in range(1, EPOCHS_SR + 1):
    losses_ep = []
    for batch in loader:
        lr_t = batch['lr'].to(device)
        hr_t = batch['hr'].to(device)
        optimizer.zero_grad()
        enhanced, _ = model(lr_t)
        if enhanced.shape != hr_t.shape:
            hr_t = F.interpolate(hr_t, size=enhanced.shape[-2:], mode='bilinear', align_corners=False)
        losses = loss_fn(enhanced, hr_t)
        losses['total'].backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        losses_ep.append(losses['total'].item())

    scheduler.step()
    mean_loss = np.mean(losses_ep)
    elapsed   = (time.time() - t0) / 60
    eta       = elapsed / epoch * (EPOCHS_SR - epoch)
    print(f'[SR] Epoch {epoch:3d}/{EPOCHS_SR} | loss {mean_loss:.5f} | {elapsed:.0f}m elapsed | {eta:.0f}m remaining')
    audit.log(epoch, epoch * len(loader), {'stage': 'image_sr', 'loss': round(float(mean_loss), 6)})

    if epoch % 20 == 0:
        torch.save(model.state_dict(), f'checkpoints/image_sr_epoch{epoch:04d}.pth')

    if mean_loss < best_loss:
        best_loss = mean_loss
        torch.save(model.state_dict(), 'checkpoints/image_sr.pth')
        print(f'  ✓ Best checkpoint updated (loss {best_loss:.5f})')

audit.save('stage1_image_sr_log.json')
print(f'\nStage 1 complete. Best loss: {best_loss:.5f}')

# Free dataset memory and disk before next stage
del dataset, loader
import shutil
shutil.rmtree('data/ImageSR_combined', ignore_errors=True)
shutil.rmtree('data/DIV2K_train_HR',   ignore_errors=True)
shutil.rmtree('data/Flickr2K',         ignore_errors=True)
print('Disk freed for next stage.')

---
# STAGE 2: Video Super-Resolution
**Dataset:** REDS (270 video clips, sharp+blur pairs)  
**Time:** ~4 hours on A100  
**Improves:** Temporal consistency, motion blur handling, flicker reduction in videos

REDS is 120 GB. Download takes ~25 minutes on Colab's fast connection.

In [ ]:
# Check disk before downloading 120 GB
os.system('df -h /content')

In [ ]:
# Download REDS dataset — sharp (for HR) and blur (for LR degradation)
# Official download requires registration; we use the academictorrents mirror
# Alternative: use gdown if you have a Google Drive link from the REDS authors

os.makedirs('data/REDS', exist_ok=True)

print('Downloading REDS train_sharp (~120 GB)...')
print('This will take ~25-35 minutes on Colab.')

# REDS official download script
!pip install gdown -q

# Sharp frames (ground truth)
!gdown --fuzzy -O data/REDS/train_sharp.zip \
    'https://drive.google.com/file/d/1YLksKtMhd2mWyVSkvhDaDLWSc1qYNCz-/view'

# If gdown fails (Google Drive rate limits), use the wget fallback:
# !wget -q --show-progress -c -O data/REDS/train_sharp.zip \
#     'https://seungjunnah.github.io/Datasets/reds.html'  # replace with actual URL

print('Extracting REDS...')
!unzip -q data/REDS/train_sharp.zip -d data/REDS/
!rm data/REDS/train_sharp.zip

reds_clips = glob.glob('data/REDS/train/train_sharp/*')
print(f'REDS ready: {len(reds_clips)} clips')

In [ ]:
import cv2
from torch.utils.data import Dataset, DataLoader
from enhancer.models.super_resolution import VideoSuperResolutionNet

class REDSDataset(Dataset):
    """Builds T-frame sequences from REDS sharp clips with synthetic degradation."""
    def __init__(self, reds_root, scale=4, patch_size=128, num_frames=5):
        self.scale = scale
        self.patch_size = patch_size
        self.num_frames = num_frames
        self.sequences = []

        # Auto-detect clip dirs — REDS has several possible layouts:
        # A: data/REDS/train/train_sharp/000/*.png
        # B: data/REDS/train_sharp/000/*.png
        # C: flat folders directly containing PNGs
        candidate_patterns = [
            os.path.join(reds_root, 'train', 'train_sharp', '*'),
            os.path.join(reds_root, 'train_sharp', '*'),
            os.path.join(reds_root, 'sharp', '*'),
            os.path.join(reds_root, 'train', '*'),
            os.path.join(reds_root, '*'),
        ]

        clip_dirs = []
        for pattern in candidate_patterns:
            candidates = [d for d in glob.glob(pattern) if os.path.isdir(d)]
            if candidates:
                has_pngs = any(glob.glob(os.path.join(d, '*.png')) for d in candidates[:3])
                if has_pngs:
                    clip_dirs = candidates
                    print(f'REDS layout detected: {pattern}')
                    break

        if not clip_dirs:
            # Recursive fallback: find all dirs with enough PNGs
            all_pngs = glob.glob(os.path.join(reds_root, '**', '*.png'), recursive=True)
            parent_dirs = list(set(os.path.dirname(p) for p in all_pngs))
            clip_dirs = [d for d in parent_dirs
                         if len(glob.glob(os.path.join(d, '*.png'))) >= num_frames]
            if clip_dirs:
                print(f'Found {len(clip_dirs)} clip dirs via recursive search')

        for clip_dir in sorted(clip_dirs):
            frames = sorted(glob.glob(os.path.join(clip_dir, '*.png')))
            if len(frames) >= num_frames:
                for i in range(len(frames) - num_frames + 1):
                    self.sequences.append(frames[i:i+num_frames])

        print(f'REDSDataset: {len(clip_dirs)} clips -> {len(self.sequences)} sequences')
        if len(self.sequences) == 0:
            raise ValueError(
                f'No sequences found under {reds_root}. '
                f'Run the structure check cell to see what was extracted.'
            )

    def __len__(self): return len(self.sequences)

    def __getitem__(self, idx):
        frame_paths = self.sequences[idx]
        frames_hr, frames_lr = [], []
        for fp in frame_paths:
            img = cv2.imread(fp)
            if img is None:
                img = np.zeros((128, 128, 3), dtype=np.uint8)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
            h, w = img.shape[:2]
            ps = self.patch_size
            if h < ps or w < ps:
                img = cv2.resize(img, (max(w, ps), max(h, ps)))
                h, w = img.shape[:2]
            y = np.random.randint(0, h - ps + 1)
            x = np.random.randint(0, w - ps + 1)
            crop_hr = img[y:y+ps, x:x+ps].copy()
            lr_ps   = ps // self.scale
            blur    = cv2.GaussianBlur(crop_hr, (3, 3), 0.5)
            crop_lr = cv2.resize(blur, (lr_ps, lr_ps), interpolation=cv2.INTER_CUBIC)
            crop_lr = np.clip(
                crop_lr + np.random.normal(
                    0, np.random.uniform(0, 10) / 255.0, crop_lr.shape
                ).astype(np.float32), 0, 1)
            frames_hr.append(torch.from_numpy(crop_hr).permute(2, 0, 1))
            frames_lr.append(torch.from_numpy(crop_lr).permute(2, 0, 1))
        return {
            'lr_seq': torch.stack(frames_lr, dim=0),
            'hr_ref': frames_hr[self.num_frames // 2],
        }

EPOCHS_VSR = 50
BATCH_VSR  = 8
LR_VSR     = 5e-5

video_model = VideoSuperResolutionNet(scale=4, num_frames=5, num_blocks=12).to(device)
vsr_dataset = REDSDataset('data/REDS', scale=4, patch_size=128, num_frames=5)
vsr_loader  = DataLoader(vsr_dataset, batch_size=BATCH_VSR, shuffle=True, num_workers=8, pin_memory=True)
vsr_opt     = optim.Adam(video_model.parameters(), lr=LR_VSR)
vsr_sched   = optim.lr_scheduler.CosineAnnealingLR(vsr_opt, T_max=EPOCHS_VSR)
vsr_loss_fn = nn.L1Loss()

print(f'Stage 2: Video SR | {len(vsr_dataset)} sequences | {EPOCHS_VSR} epochs | batch={BATCH_VSR}')


In [ ]:
best_vsr_loss = float('inf')
t0 = time.time()

for epoch in range(1, EPOCHS_VSR + 1):
    losses_ep = []
    video_model.train()
    for batch in vsr_loader:
        lr_seq = batch['lr_seq'].to(device)   # (B, T, C, lH, lW)
        hr_ref = batch['hr_ref'].to(device)   # (B, C, H, W)
        vsr_opt.zero_grad()
        enhanced, _ = video_model(lr_seq, flows=None)
        if enhanced.shape != hr_ref.shape:
            hr_ref = F.interpolate(hr_ref, size=enhanced.shape[-2:], mode='bilinear', align_corners=False)
        loss = vsr_loss_fn(enhanced, hr_ref)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(video_model.parameters(), 1.0)
        vsr_opt.step()
        losses_ep.append(loss.item())

    vsr_sched.step()
    mean_loss = np.mean(losses_ep)
    elapsed   = (time.time() - t0) / 60
    eta       = elapsed / epoch * (EPOCHS_VSR - epoch)
    print(f'[VSR] Epoch {epoch:3d}/{EPOCHS_VSR} | loss {mean_loss:.5f} | {elapsed:.0f}m elapsed | {eta:.0f}m remaining')

    if epoch % 10 == 0:
        torch.save(video_model.state_dict(), f'checkpoints/video_sr_epoch{epoch:04d}.pth')

    if mean_loss < best_vsr_loss:
        best_vsr_loss = mean_loss
        torch.save(video_model.state_dict(), 'checkpoints/video_sr.pth')
        print(f'  ✓ Best video SR checkpoint (loss {best_vsr_loss:.5f})')

print(f'\nStage 2 complete. Best loss: {best_vsr_loss:.5f}')

# Free REDS disk space
del vsr_dataset, vsr_loader
shutil.rmtree('data/REDS', ignore_errors=True)
print('REDS data freed.')

---
# STAGE 3: Deblurring
**Dataset:** GoPro Sharp/Blur pairs  
**Time:** ~1 hour on A100  
**Improves:** Motion blur removal without hallucination

In [ ]:
print('Downloading GoPro dataset (~6 GB)...')
!gdown --fuzzy -O data/GoPro.zip \
    'https://drive.google.com/file/d/1H0PIXvJH4c40pk7ou6nAwoxuR4Qh_Sa2/view'
!unzip -q data/GoPro.zip -d data/
!rm data/GoPro.zip
print(f'GoPro ready: {len(glob.glob("data/GoPro/train/*/blur/*.png"))} blur frames')

In [ ]:
from enhancer.models.deblur import DeblurNet

class GoProDataset(Dataset):
    """Loads GoPro blur/sharp pairs for deblur training."""
    def __init__(self, root, patch_size=256, split='train'):
        self.patch_size = patch_size
        self.pairs = []
        for seq in glob.glob(os.path.join(root, split, '*')):
            blur_frames  = sorted(glob.glob(os.path.join(seq, 'blur',  '*.png')))
            sharp_frames = sorted(glob.glob(os.path.join(seq, 'sharp', '*.png')))
            self.pairs.extend(zip(blur_frames, sharp_frames))
        print(f'GoProDataset ({split}): {len(self.pairs)} pairs')

    def __len__(self): return len(self.pairs)

    def __getitem__(self, idx):
        blur_path, sharp_path = self.pairs[idx]
        blur  = cv2.cvtColor(cv2.imread(blur_path),  cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        sharp = cv2.cvtColor(cv2.imread(sharp_path), cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        h, w  = blur.shape[:2]
        ps    = self.patch_size
        y = np.random.randint(0, max(h - ps, 1))
        x = np.random.randint(0, max(w - ps, 1))
        blur_c  = blur[y:y+ps,  x:x+ps].copy()
        sharp_c = sharp[y:y+ps, x:x+ps].copy()
        if np.random.rand() > 0.5:
            blur_c  = np.fliplr(blur_c).copy()
            sharp_c = np.fliplr(sharp_c).copy()
        return {
            'blur':  torch.from_numpy(blur_c).permute(2, 0, 1),
            'sharp': torch.from_numpy(sharp_c).permute(2, 0, 1),
        }

EPOCHS_DEBLUR = 50
BATCH_DEBLUR  = 8
LR_DEBLUR     = 1e-4

deblur_model   = DeblurNet().to(device)
deblur_dataset = GoProDataset('data/GoPro', patch_size=256)
deblur_loader  = DataLoader(deblur_dataset, batch_size=BATCH_DEBLUR, shuffle=True, num_workers=8, pin_memory=True)
deblur_opt     = optim.Adam(deblur_model.parameters(), lr=LR_DEBLUR)
deblur_sched   = optim.lr_scheduler.CosineAnnealingLR(deblur_opt, T_max=EPOCHS_DEBLUR)
deblur_loss_fn = nn.L1Loss()

print(f'Stage 3: Deblur | {len(deblur_dataset)} pairs | {EPOCHS_DEBLUR} epochs')

In [ ]:
best_deblur_loss = float('inf')
t0 = time.time()

for epoch in range(1, EPOCHS_DEBLUR + 1):
    losses_ep = []
    deblur_model.train()
    for batch in deblur_loader:
        blur_t  = batch['blur'].to(device)
        sharp_t = batch['sharp'].to(device)
        deblur_opt.zero_grad()
        deblurred, _ = deblur_model(blur_t, strength=1.0)
        loss = deblur_loss_fn(deblurred, sharp_t)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(deblur_model.parameters(), 1.0)
        deblur_opt.step()
        losses_ep.append(loss.item())

    deblur_sched.step()
    mean_loss = np.mean(losses_ep)
    elapsed   = (time.time() - t0) / 60
    eta       = elapsed / epoch * (EPOCHS_DEBLUR - epoch)
    print(f'[Deblur] Epoch {epoch:3d}/{EPOCHS_DEBLUR} | loss {mean_loss:.5f} | {elapsed:.0f}m elapsed | {eta:.0f}m remaining')

    if mean_loss < best_deblur_loss:
        best_deblur_loss = mean_loss
        torch.save(deblur_model.state_dict(), 'checkpoints/deblur.pth')
        print(f'  ✓ Best deblur checkpoint (loss {best_deblur_loss:.5f})')

print(f'\nStage 3 complete. Best loss: {best_deblur_loss:.5f}')
del deblur_dataset, deblur_loader
shutil.rmtree('data/GoPro', ignore_errors=True)
print('GoPro data freed.')

---
# STAGE 4: Joint fine-tuning
**Dataset:** Small DIV2K sample (re-downloaded, 100 images)  
**Time:** ~30 min  
**Purpose:** Calibrate all models together so outputs are consistent

In [ ]:
# Re-download a small DIV2K sample for joint fine-tuning
!wget -q --show-progress -O data/DIV2K_valid_HR.zip \
    http://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_valid_HR.zip
!unzip -q data/DIV2K_valid_HR.zip -d data/
!rm data/DIV2K_valid_HR.zip
print(f'Validation set: {len(glob.glob("data/DIV2K_valid_HR/*.png"))} images')

In [ ]:
# Joint fine-tune: SR + Deblur together at very low LR
EPOCHS_JOINT = 20
LR_JOINT     = 2e-6

model.train()
deblur_model.train()

joint_params  = list(model.parameters()) + list(deblur_model.parameters())
joint_opt     = optim.Adam(joint_params, lr=LR_JOINT)
joint_dataset = SyntheticDegradationDataset('data/DIV2K_valid_HR', scale=4, patch_size=128)
joint_loader  = DataLoader(joint_dataset, batch_size=8, shuffle=True, num_workers=4, pin_memory=True)

print(f'Stage 4: Joint fine-tune | {len(joint_dataset)} samples | {EPOCHS_JOINT} epochs')
t0 = time.time()

for epoch in range(1, EPOCHS_JOINT + 1):
    losses_ep = []
    for batch in joint_loader:
        lr_t = batch['lr'].to(device)
        hr_t = batch['hr'].to(device)
        joint_opt.zero_grad()

        # SR pass
        enhanced, _ = model(lr_t)
        if enhanced.shape != hr_t.shape:
            hr_t_r = F.interpolate(hr_t, size=enhanced.shape[-2:], mode='bilinear', align_corners=False)
        else:
            hr_t_r = hr_t

        # Deblur pass on SR output
        deblurred, _ = deblur_model(enhanced, strength=0.5)

        loss = F.l1_loss(deblurred, hr_t_r)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(joint_params, 0.5)
        joint_opt.step()
        losses_ep.append(loss.item())

    mean_loss = np.mean(losses_ep)
    print(f'[Joint] Epoch {epoch:2d}/{EPOCHS_JOINT} | loss {mean_loss:.5f} | {(time.time()-t0)/60:.0f}m')

# Save final joint checkpoints
torch.save(model.state_dict(),        'checkpoints/image_sr.pth')
torch.save(deblur_model.state_dict(), 'checkpoints/deblur.pth')
print('\nStage 4 complete. Final checkpoints saved.')

---
## Step 5: Test all models on a sample image

In [ ]:
from IPython.display import Image as IPImage, display

test_imgs = glob.glob('data/DIV2K_valid_HR/*.png')[:1]
if test_imgs:
    test_img = test_imgs[0]
    print(f'Testing on: {test_img}')
    !python scripts/enhance_cli.py image {test_img} /tmp/test_enhanced.png \
        --mode balanced --checkpoint-dir checkpoints --width 1920 --height 1080
    print('Original:')
    display(IPImage(test_img, width=600))
    print('Enhanced:')
    display(IPImage('/tmp/test_enhanced.png', width=600))

---
## Step 6: Download all checkpoints

In [ ]:
import zipfile as zf
from google.colab import files

# Zip all checkpoints together
with zf.ZipFile('all_checkpoints.zip', 'w', zf.ZIP_DEFLATED) as z:
    for f in glob.glob('checkpoints/*.pth'):
        z.write(f)
    for f in glob.glob('training_logs/*.json'):
        z.write(f)

size_mb = os.path.getsize('all_checkpoints.zip') / 1e6
print(f'Downloading all_checkpoints.zip ({size_mb:.0f} MB)...')
files.download('all_checkpoints.zip')

In [ ]:
# Or download individual checkpoints
for ckpt in ['checkpoints/image_sr.pth', 'checkpoints/video_sr.pth', 'checkpoints/deblur.pth']:
    if os.path.exists(ckpt):
        print(f'Downloading {ckpt}...')
        files.download(ckpt)

---
## After downloading

Extract `all_checkpoints.zip` into your local `checkpoints/` folder:
```
video_image_enhancer/
  checkpoints/
    image_sr.pth   ← main SR model
    video_sr.pth   ← video SR model
    deblur.pth     ← deblur model
```

Then test locally:
```powershell
python scripts/enhance_cli.py image photo.jpg out.png --mode balanced --checkpoint-dir checkpoints
```

And deploy:
```powershell
python -m enhancer.api.server
# Open web/index.html in your browser
```

See `docs/web_deployment.md` for deploying to a live website.